# Step 0 — Task playbook feasibility audit

Answers one question: **can we derive task playbooks** (a canonical task list, each with a lifecycle phase and a median day-offset from project start, per project type) **from our completed projects** — instead of hand-authoring them?

This notebook is **read-only**. Nothing is written to AGOL.

What it reports:
1. Completed projects per type — do we have enough examples per type?
2. Task linkage — are completed projects actually broken into tasks?
3. Offset computability — can we turn task dates into reliable relative due-dates?
4. Phase signal — is phase_requirements populated enough to bucket tasks by lifecycle phase?
5. Draft playbook for the top types — the eyeball test: does a real, repeatable shape emerge?

Decision rule mirrors the estimate-calibration audit: a type with n>=6 completed projects is High confidence, 3–5 Medium, <3 Insufficient.

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
import pandas as pd
import datetime as dt

gis = GIS("home")
print(f"Authenticated as: {gis.users.me.username} @ {gis.url}")

## Config

Same source service as the calibration audit (datateam_portfolio_v2): layer 0 = projects, layer 1 = tasks. Bump WINDOW_MONTHS up (or to 9999 for all-time) if recent volume looks thin — playbooks benefit from more history than calibration did.

In [ ]:
SOURCE_TASKS_URL    = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer/1"
SOURCE_PROJECTS_URL = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer/0"

# How far back to look (24mo default; raise if thin).
WINDOW_MONTHS = 24

# A "type" is what we build one playbook per. Start with category; switch to
# ["category", "project_size"] if categories are too coarse AND you have volume.
TYPE_KEYS = ["category"]

# A derived playbook needs fewer examples than a calibration multiplier: a
# repeated shape across 6+ completed projects is solid, 3-5 suggestive, <3 guesswork.
N_HIGH = 6
N_MED  = 3

# A completed project needs at least this many tasks to tell us anything.
MIN_TASKS = 3

# How many top types to print a draft playbook for.
TOP_TYPES = 3

# Lifecycle phase names (match the app's PROJECT_PHASES short names).
PHASE_NAMES = {0:"0 Intake", 1:"1 Kickoff", 2:"2 Data prep", 3:"3 Build", 4:"4 QA",
               5:"5 Design review", 6:"6 Partner review", 7:"7 Launch", 8:"8 Acceptance"}

CUTOFF = pd.Timestamp(dt.datetime.utcnow()) - pd.DateOffset(months=WINDOW_MONTHS)
print(f"Window: last {WINDOW_MONTHS} months  (cutoff = {CUTOFF.date()})")
print(f"Type key(s): {TYPE_KEYS}")
print(f"Per-type confidence: n>={N_HIGH} High, {N_MED}-{N_HIGH-1} Medium, <{N_MED} Insufficient")

## 1 — Pull projects (defensive)

Prints the layer's real field list first, then pulls everything and filters in pandas, so a schema drift surfaces as a clear message instead of a generic 400.

In [ ]:
proj_layer = FeatureLayer(SOURCE_PROJECTS_URL, gis=gis)

print("Projects layer fields:")
for f in proj_layer.properties.fields:
    print(f"  {f['name']:28s} {f['type']}")
print("")

pfset = proj_layer.query(where="1=1", out_fields="*", return_geometry=False)
projects = pd.DataFrame([f.attributes for f in pfset.features])
print(f"Pulled {len(projects)} total projects.")

proj_required = ["status", "start_date", "category", "project_number"]
proj_missing = [c for c in proj_required if c not in projects.columns]
if proj_missing:
    print(f"WARNING: missing expected project fields: {proj_missing}")
    for m in proj_missing:
        norm = m.replace("_", "").lower()
        hits = [c for c in projects.columns if norm in c.replace("_", "").lower()]
        print(f"  {m:16s} -> {hits}")
    raise KeyError(f"Update field name(s): {proj_missing}")

for c in ["start_date", "end_date", "actual_end"]:
    if c in projects.columns:
        projects[c + "_dt"] = pd.to_datetime(projects[c], unit="ms", errors="coerce")

projects["category"] = projects["category"].fillna("(uncategorized)")
if "project_size" in projects.columns:
    projects["project_size"] = projects["project_size"].fillna("(no size)")

if "actual_end_dt" in projects.columns:
    completed = projects[(projects["status"] == "Complete") & (projects["actual_end_dt"].notna()) & (projects["actual_end_dt"] >= CUTOFF)].copy()
else:
    print("(no actual_end column — filtering on status only)")
    completed = projects[projects["status"] == "Complete"].copy()
completed["project_number_str"] = completed["project_number"].astype(str)
print(f"Completed projects in window: {len(completed)}")

## 2 — Pull tasks (defensive)

phase_requirements, working_due, tool, category, and assignee are treated as optional — the audit degrades gracefully if any are absent.

In [ ]:
tasks_layer = FeatureLayer(SOURCE_TASKS_URL, gis=gis)

print("Tasks layer fields:")
for f in tasks_layer.properties.fields:
    print(f"  {f['name']:28s} {f['type']}")
print("")

tfset = tasks_layer.query(where="1=1", out_fields="*", return_geometry=False)
tasks = pd.DataFrame([f.attributes for f in tfset.features])
print(f"Pulled {len(tasks)} total tasks.")

task_required = ["project_number", "title"]
task_missing = [c for c in task_required if c not in tasks.columns]
if task_missing:
    print(f"WARNING: missing expected task fields: {task_missing}")
    for m in task_missing:
        norm = m.replace("_", "").lower()
        hits = [c for c in tasks.columns if norm in c.replace("_", "").lower()]
        print(f"  {m:16s} -> {hits}")
    raise KeyError(f"Update field name(s): {task_missing}")

for c in ["due", "working_due", "start", "actual_end"]:
    if c in tasks.columns:
        tasks[c + "_dt"] = pd.to_datetime(tasks[c], unit="ms", errors="coerce")

has_phase = "phase_requirements" in tasks.columns
print(f"phase_requirements present: {has_phase}")
for opt in ["category", "tool", "assignee"]:
    if opt not in tasks.columns:
        tasks[opt] = "(none)"
    else:
        tasks[opt] = tasks[opt].fillna("(none)")

## 3 — Completed projects per type

How many playbooks could we derive, and at what confidence? This is the binding constraint — a type needs a handful of completed examples before a "shape" is real and not noise.

In [ ]:
def tier(n):
    if n >= N_HIGH: return "High"
    if n >= N_MED:  return "Medium"
    return "Insufficient"

completed["_type"] = completed[TYPE_KEYS].astype(str).agg(" / ".join, axis=1)
by_type = completed.groupby("_type").size().rename("n_projects").reset_index().sort_values("n_projects", ascending=False).reset_index(drop=True)
by_type["confidence"] = by_type["n_projects"].apply(tier)
print(by_type.to_string(index=False))
print("")

n_high = (by_type["confidence"] == "High").sum()
n_med  = (by_type["confidence"] == "Medium").sum()
n_low  = (by_type["confidence"] == "Insufficient").sum()
print(f"Types: {n_high} High, {n_med} Medium, {n_low} Insufficient")
if n_high >= 3:
    print("  [OK] Several types have enough completed projects to derive a real playbook.")
elif n_high + n_med >= 3:
    print("  [~] Derive for the Medium+ types with a 'draft from thin data' caveat.")
else:
    print("  [NO] Too few completed projects per type. Seed playbooks by hand for now; revisit as history grows.")

## 4 — Task linkage

A playbook can only learn from projects that were actually broken into tasks. Reports how many completed projects clear MIN_TASKS, and the typical task count.

In [ ]:
comp_nums = set(completed["project_number_str"])
tasks["project_number_str"] = tasks["project_number"].astype(str)
ctasks = tasks[tasks["project_number_str"].isin(comp_nums)].copy()
print(f"Tasks on completed projects: {len(ctasks)}")

# map (not merge) so re-running this cell stays idempotent — a re-merge would
# collide the n_tasks column into n_tasks_x / n_tasks_y.
per_proj = ctasks.groupby("project_number_str").size()
completed["n_tasks"] = completed["project_number_str"].map(per_proj).fillna(0).astype(int)

n_comp = len(completed)
n_with = (completed["n_tasks"] >= MIN_TASKS).sum()
print(f"Completed projects with >= {MIN_TASKS} tasks: {n_with} / {n_comp} ({100*n_with/max(n_comp,1):.0f}%)")
print(f"Median tasks per completed project: {completed['n_tasks'].median():.0f}")
print(f"Mean tasks per completed project:   {completed['n_tasks'].mean():.1f}")
if n_comp and n_with / n_comp >= 0.5:
    print("  [OK] Most completed projects are tasked — a playbook has material to learn from.")
elif n_comp and n_with / n_comp >= 0.25:
    print("  [~] Many projects are thinly tasked; playbooks will lean on the better-documented ones.")
else:
    print("  [NO] Projects aren't consistently broken into tasks — mining won't have much to chew on.")

## 5 — Offset computability

Each playbook task carries a day-offset from the project start. Target date = first available of due / working_due / actual_end / start, minus the project's start_date. Reports how many tasks yield a sane offset.

In [ ]:
# Look up each task's project start date by map (not merge) so the cell is re-run safe.
start_map = completed.set_index("project_number_str")["start_date_dt"]
ctasks["start_date_dt"] = pd.to_datetime(ctasks["project_number_str"].map(start_map), errors="coerce")

# Coalesce each task's target date from the first available date column. Vectorized
# so the result keeps datetime64 dtype — an object-dtype Series can't be subtracted
# from a DatetimeArray.
target = pd.Series(pd.NaT, index=ctasks.index, dtype="datetime64[ns]")
for c in ["due_dt", "working_due_dt", "actual_end_dt", "start_dt"]:
    if c in ctasks.columns:
        target = target.fillna(ctasks[c])
ctasks["target_dt"] = target
ctasks["offset_days"] = (ctasks["target_dt"] - ctasks["start_date_dt"]).dt.days

n_ct = len(ctasks)
n_off = ctasks["offset_days"].notna().sum()
print(f"Tasks with a computable offset: {n_off} / {n_ct} ({100*n_off/max(n_ct,1):.0f}%)")
print(f"  Negative (task due before project start): {(ctasks['offset_days'] < 0).sum()}")
print(f"  Over 2 years (likely bad data):           {(ctasks['offset_days'] > 730).sum()}")
ok = ctasks[ctasks["offset_days"].between(0, 730)]
if len(ok):
    print(f"  Sane-offset distribution: median {ok['offset_days'].median():.0f}d, p25 {ok['offset_days'].quantile(.25):.0f}d, p75 {ok['offset_days'].quantile(.75):.0f}d, max {ok['offset_days'].max():.0f}d")
if n_ct and n_off / n_ct >= 0.5:
    print("  [OK] Offsets computable for most tasks — relative due-dates will hold up.")
else:
    print("  [~] Many tasks lack usable dates; offsets will come from a smaller subset.")

## 6 — Phase signal

If phase_requirements is well populated, playbook tasks bucket cleanly by lifecycle phase (the prefix Px in IDs like P3_DEMOS). If sparse, fall back to task category / offset order, or let an AI pass infer phase.

In [ ]:
def phase_nums(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    out = []
    for token in str(val).split(","):
        token = token.strip()
        if not token or token[0].lower() != "p":
            continue
        digits = ""
        for ch in token[1:]:
            if ch.isdigit():
                digits += ch
            else:
                break
        if digits:
            out.append(int(digits))
    return sorted(set(out))

if has_phase:
    ctasks["phases"] = ctasks["phase_requirements"].apply(phase_nums)
    ctasks["primary_phase"] = ctasks["phases"].apply(lambda ps: ps[0] if ps else -1)
    n_phased = (ctasks["primary_phase"] >= 0).sum()
    print(f"Tasks with a phase: {n_phased} / {len(ctasks)} ({100*n_phased/max(len(ctasks),1):.0f}%)")
    dist = ctasks[ctasks["primary_phase"] >= 0]["primary_phase"].map(PHASE_NAMES).value_counts()
    print("Tasks by primary phase:")
    print(dist.to_string())
    if len(ctasks) and n_phased / len(ctasks) >= 0.4:
        print("  [OK] Enough phase coverage to bucket playbook tasks by lifecycle phase.")
    else:
        print("  [~] Phase coverage thin — bucket by category/offset, or infer phase with AI.")
else:
    ctasks["primary_phase"] = -1
    print("phase_requirements not on this layer — bucket by category/offset, or infer phase with AI.")

## 7 — Draft playbook for the top types

The eyeball test. For each top type: the phase skeleton (tasks per phase, % of the type's projects that include that phase, median offset), a title-repeatability metric (high = titles cluster by themselves; low = an AI canonicalization pass is needed), and the most common tasks with their median offsets.

In [ ]:
def norm_title(s):
    s = str(s).lower().strip()
    s = " ".join(s.split())
    return s.rstrip(".:;-_ ")

type_by_num = completed.set_index("project_number_str")["_type"]
ctasks["_type"] = ctasks["project_number_str"].map(type_by_num)
ctasks["norm_title"] = ctasks["title"].apply(norm_title)

top = by_type.head(TOP_TYPES)["_type"].tolist()
for tp in top:
    sub = ctasks[(ctasks["_type"] == tp) & ctasks["offset_days"].between(0, 730)]
    nproj = completed[completed["_type"] == tp]["project_number_str"].nunique()
    print("")
    print(f"================  {tp}   ({nproj} completed projects, {len(sub)} dated tasks)  ================")
    if not len(sub):
        print("  (no dated tasks to aggregate)")
        continue
    rows = []
    for ph in sorted(sub["primary_phase"].unique()):
        php = sub[sub["primary_phase"] == ph]
        label = PHASE_NAMES.get(ph, "(unphased)") if ph >= 0 else "(unphased)"
        examples = ", ".join(php["norm_title"].value_counts().head(3).index.tolist())
        rows.append({
            "phase": label,
            "n_tasks": len(php),
            "pct_projects": f"{100*php['project_number_str'].nunique()/max(nproj,1):.0f}%",
            "median_off_d": int(php["offset_days"].median()),
            "examples": examples[:70],
        })
    print(pd.DataFrame(rows).to_string(index=False))
    vc = sub["norm_title"].value_counts()
    top10_share = vc.head(10).sum() / max(len(sub), 1)
    print(f"  Title repeatability: top-10 titles cover {100*top10_share:.0f}% of tasks ({sub['norm_title'].nunique()} distinct / {len(sub)} tasks)")
    print("  Most common tasks (count, median offset, title):")
    for title, cnt in vc.head(8).items():
        med = int(sub[sub["norm_title"] == title]["offset_days"].median())
        print(f"    {cnt:3d}x  +{med:>3}d   {title[:60]}")

## 8 — Decision

Read the section verdicts together:

| Signal | Section | What it tells you |
|---|---|---|
| Types with n>=6 completed | 3 | How many playbooks you can derive with confidence |
| % projects with >=3 tasks | 4 | Whether projects are tasked enough to learn from |
| % tasks with a computable offset | 5 | Whether relative due-dates will be reliable |
| % tasks with a phase | 6 | Bucket by phase, or fall back to category/AI |
| Title repeatability | 7 | High -> group by title directly; Low -> AI canonicalization needed |

Build path:
- **Green across the board** -> derive draft playbooks for the High-confidence types: tasks grouped by phase, median offset, kept if they appear in a majority of that type's projects. A lead reviews/edits in the template editor before publishing — never auto-publish.
- **Good volume, low title repeatability** -> hybrid: stats give phase + offset; an AI pass canonicalizes task names from the raw titles (build on the existing AI intake / phase-assignment / suggest-alignment).
- **Thin per-type volume** -> ship the hand-authored templates now (the mockup seeds) and expose a "Suggest from history" button per type that lights up only once that type clears n>=6.

Drop the outputs back into the conversation and we'll pick the path. Nothing here writes to AGOL.